## Imports

In [1]:
from pathlib import Path
import sys
import os

In [2]:
import numpy as np

import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule

In [3]:
project_working_dir = str(Path(sys.path[0]).parent)
sys.path += [project_working_dir]
os.chdir(project_working_dir)

In [4]:
from src.utils.file_io import (
    read_file_str,
    show_formatted_cpp,
    replace_constants_in_kernel,
)

In [5]:
!cl

usage: cl [ option... ] filename... [ /link linkoption... ]


Microsoft (R) C/C++ Optimizing Compiler Version 19.43.34810 for x64
Copyright (C) Microsoft Corporation.  All rights reserved.



## Create sample data

In [6]:
vertices = np.array(
    [
        [0, 0, 0],
        [1, 0, 0],
        [2, 0, 0],
    ],
    dtype=np.float32,
)

In [7]:
other_vertices = np.array(
    [
        [0, 0, 0],
        [2, 0, 0],
        [7, 0, 0],
    ],
    dtype=np.float32,
)

In [8]:
sewing_indices = np.array([[0, 0], [1, 1], [2, 2]], dtype=np.uint32)

Re-index the to vertices

In [9]:
nr_from_vertices = len(vertices)

In [10]:
sewing_indices[:, 1] += nr_from_vertices

In [11]:
vertices = np.vstack([vertices, other_vertices])

In [12]:
sewing_indices

array([[0, 3],
       [1, 4],
       [2, 5]], dtype=uint32)

In [13]:
accelerations = np.array(
    [
        [-1, 0, 0],
        [-1, 0, 0],
        [-1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
    ],
    dtype=np.float32,
)

## Cuda Parameters

In [14]:
BLOCK_SIZE = 1024
NR_BLOCKS = (len(vertices) + BLOCK_SIZE - 1) // BLOCK_SIZE

## Compile cuda kernel

In [15]:
cuda_code = read_file_str("./src/cuda_kernels/apply_sewing_constraints.cu")

In [16]:
parameter_updates = {
    "TIME_DELTA": 1,
    "SEWING_MAX_ADJUSTMENT": 0.5,
    "SEWING_FORCE_MULTIPLIER": 1,
}

In [17]:
cuda_code = replace_constants_in_kernel(cuda_code, parameter_updates)

In [18]:
show_formatted_cpp(cuda_code)

In [19]:
mod = SourceModule(cuda_code)

## Set-up memory for running kernel

In [20]:
apply_sewing_constraints = mod.get_function("apply_sewing_constraints")

In [21]:
nr_sewing = np.uint32(len(sewing_indices))

### Allocate memory to gpu

In [22]:
assert sewing_indices.flatten().flags["C_CONTIGUOUS"]
assert vertices.flatten().flags["C_CONTIGUOUS"]
assert accelerations.flatten().flags["C_CONTIGUOUS"]

In [23]:
vertices_gpu = cuda.mem_alloc(vertices.nbytes)
accelerations_gpu = cuda.mem_alloc(accelerations.nbytes)
sewing_indices_gpu = cuda.mem_alloc(sewing_indices.nbytes)

In [24]:
cuda.memcpy_htod(vertices_gpu, vertices.flatten())
cuda.memcpy_htod(accelerations_gpu, accelerations.flatten())
cuda.memcpy_htod(sewing_indices_gpu, sewing_indices.flatten())

## Create function

In [25]:
def sewing_kernel():
    apply_sewing_constraints(
        vertices_gpu,
        accelerations_gpu,
        sewing_indices_gpu,
        nr_sewing,
        block=(BLOCK_SIZE, 1, 1),
        grid=(NR_BLOCKS, 1, 1),
    )

## Check output is as expected

In [26]:
sewing_kernel()

In [27]:
cuda.memcpy_dtoh(vertices, vertices_gpu)
cuda.memcpy_dtoh(accelerations, accelerations_gpu)

In [28]:
vertices

array([[0. , 0. , 0. ],
       [1.5, 0. , 0. ],
       [2.5, 0. , 0. ],
       [0. , 0. , 0. ],
       [1.5, 0. , 0. ],
       [6.5, 0. , 0. ]], dtype=float32)

In [29]:
accelerations

array([[-1. ,  0. ,  0. ],
       [-1.5,  0. ,  0. ],
       [-1.5,  0. ,  0. ],
       [ 1. ,  0. ,  0. ],
       [ 1.5,  0. ,  0. ],
       [ 1.5,  0. ,  0. ]], dtype=float32)

Two close vertices move by maximum sewing adjustment.

In [30]:
assert np.isclose(vertices[1, 0], vertices[4, 0])

Cannot move more than maximum sewing adjustment

In [31]:
assert np.isclose(vertices[5, 0] - vertices[2, 0], 4)

Vertices that are already close do not need to move together.

In [32]:
assert np.isclose(accelerations[0, 0], -1)
assert np.isclose(accelerations[3, 0], 1)

## Profile function

At this point computation is so small that it seems fixed.

In [33]:
%timeit sewing_kernel()

11 µs ± 280 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
